In [17]:
from field_config import config
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from time import perf_counter

In [18]:
import importlib
import synthetic_fields

importlib.reload(synthetic_fields)

from synthetic_fields import generate_cube, save_cube

In [9]:
print(config)

{'field_synth': 'velocity', 'model_gs': 'stable', 'alpha': 1.5, 'shape': (64, 64, 64), 'spacing': (1.0, 1.0, 1.0), 'len_scale': (16.0, 16.0, 16.0), 'angles': (0.0, 0.0, 0.0), 'seed': 19880127}


In [3]:
from copy import deepcopy

models = ("exponential", "gaussian", "stable")

field_seeds = {
    "velocity": 19880127,
    "density": 19970221,
}

configurations = []

for model_name in models:
    for field_role, seed in field_seeds.items():
        cube_config = deepcopy(config)

        cube_config.update(
            model_gs=model_name,
            field_synth=field_role,
            seed=seed,
        )

        # Only Stable receives a configurable alpha.
        if model_name == "stable":
            cube_config["alpha"] = config.get("alpha", 1.5)
        else:
            cube_config.pop("alpha", None)

        configurations.append(cube_config)

In [4]:
for index, cube_config in enumerate(configurations, start=1):
    print(
        f"{index}. "
        f"{cube_config['model_gs']:12s} "
        f"{cube_config['field_synth']:8s} "
        f"shape={cube_config['shape']} "
        f"seed={cube_config['seed']} "
        f"alpha={cube_config.get('alpha', 'fixed')}"
    )

1. exponential  velocity shape=(64, 64, 64) seed=19880127 alpha=fixed
2. exponential  density  shape=(64, 64, 64) seed=19970221 alpha=fixed
3. gaussian     velocity shape=(64, 64, 64) seed=19880127 alpha=fixed
4. gaussian     density  shape=(64, 64, 64) seed=19970221 alpha=fixed
5. stable       velocity shape=(64, 64, 64) seed=19880127 alpha=1.5
6. stable       density  shape=(64, 64, 64) seed=19970221 alpha=1.5


In [5]:
# Dedicated folder for this six-cube test.
output_dir = "cubes/batch_test_64"

# Set True only when deliberately replacing a previous batch.
overwrite = True

saved_paths = []

In [6]:


for index, cube_config in enumerate(configurations, start=1):
    start = perf_counter()

    print(
        f"[{index}/{len(configurations)}] Generating "
        f"{cube_config['model_gs']} "
        f"{cube_config['field_synth']}...",
        flush=True,
    )

    field_3d, grid, model, m3d = generate_cube(cube_config)

    fits_path = save_cube(
        field_3d,
        cube_config,
        m3d,
        output_dir=output_dir,
        overwrite=overwrite,
    )

    saved_paths.append(fits_path)

    print(
        f"Saved: {fits_path.name} "
        f"({perf_counter() - start:.1f} s)",
        flush=True,
    )

    # Release this cube before generating the next one.
    del field_3d, grid, model

print(f"\nCompleted: {len(saved_paths)} cubes saved.")

[1/6] Generating exponential velocity...
Saved: exponential_3d_velocity_N64x64x64_seed19880127.fits (3.4 s)
[2/6] Generating exponential density...
Saved: exponential_3d_density_N64x64x64_seed19970221.fits (3.4 s)
[3/6] Generating gaussian velocity...
Saved: gaussian_3d_velocity_N64x64x64_seed19880127.fits (3.3 s)
[4/6] Generating gaussian density...
Saved: gaussian_3d_density_N64x64x64_seed19970221.fits (3.2 s)
[5/6] Generating stable velocity...
Saved: stable_3d_velocity_N64x64x64_seed19880127.fits (3.4 s)
[6/6] Generating stable density...
Saved: stable_3d_density_N64x64x64_seed19970221.fits (3.5 s)

Completed: 6 cubes saved.


In [7]:
from astropy.io import fits

print(
    f"{'Model':12s} {'Field':8s} {'M3D':>5s} "
    f"{'Min':>10s} {'Max':>10s} {'Mean':>10s} {'Std':>10s}"
)

for path in saved_paths:
    header = fits.getheader(path)

    print(
        f"{header['MODEL']:12s} "
        f"{header['FIELD']:8s} "
        f"{header['M3D']:5.2f} "
        f"{header['DATAMIN']:10.4f} "
        f"{header['DATAMAX']:10.4f} "
        f"{header['MEAN']:10.4f} "
        f"{header['STD']:10.4f}"
    )

Model        Field      M3D        Min        Max       Mean        Std
exponential  velocity  1.00    -3.4632     3.1256    -0.3626     0.7976
exponential  density   1.00    -2.9442     3.4235     0.1190     0.8087
gaussian     velocity  2.00    -2.6302     2.4502    -0.1659     0.9452
gaussian     density   2.00    -3.2418     2.3904    -0.2377     0.8619
stable       velocity  1.50    -3.7811     3.5600     0.1442     1.0050
stable       density   1.50    -3.6706     3.7578    -0.0231     0.8707


In [19]:
from copy import deepcopy

comparison_configs = []

for alpha in (1.0, 2.0):
    for field_role, seed in field_seeds.items():
        cube_config = deepcopy(config)

        cube_config.update(
            model_gs="stable",
            alpha=alpha,
            field_synth=field_role,
            seed=seed,
        )

        comparison_configs.append(cube_config)

comparison_paths = []

for cube_config in comparison_configs:
    print(
        f"Generating Stable alpha={cube_config['alpha']:g}, "
        f"{cube_config['field_synth']}...",
        flush=True,
    )

    field_3d, grid, model, m3d = generate_cube(cube_config)

    path = save_cube(
        field_3d,
        cube_config,
        m3d,
        output_dir=output_dir,
        overwrite=False,
    )

    comparison_paths.append(path)
    print(f"Saved: {path.name}", flush=True)

    del field_3d, grid, model

Generating Stable alpha=1, velocity...
Saved: stable_alpha1_3d_velocity_N64x64x64_seed19880127.fits
Generating Stable alpha=1, density...
Saved: stable_alpha1_3d_density_N64x64x64_seed19970221.fits
Generating Stable alpha=2, velocity...
Saved: stable_alpha2_3d_velocity_N64x64x64_seed19880127.fits
Generating Stable alpha=2, density...
Saved: stable_alpha2_3d_density_N64x64x64_seed19970221.fits
